# Shot Characteristics Classification

## Overview

A campaign produces thousands of shots, most of which nobody will ever open
individually. What makes them tractable is a per-shot record: a handful of
features extracted the same way for every shot, a class label with the evidence
behind it, and a review state saying whether a human has looked.

This notebook builds that record for the three shots packaged with VAFT, using
the classification and timing APIs the library already has, and writes the
summary table a campaign-level analysis would read. It runs offline; the
database-backed path over a whole campaign is shown but gated.

## Pipeline context

Downstream of raw-signal loading (`vest_raw_signal_sql_database.ipynb`) and
diagnostics processing (`magnetic_diagnostics_processing.ipynb`), and of the
startup analysis in `eddy_current_calculation_and_startup_analysis.ipynb`. Its
output is the aggregation layer that reporting and comparison pages read.

## Representative signal inventory

Classification reads four signals: the plasma current, the fill pressure, the
H-alpha light, and the reconstructed equilibrium. What a shot carries decides
what can be said about it, so the inventory comes first.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import vaft
from vaft.omas.plasma_timing import plasma_timing
from vaft.process.equilibrium import as_equilibrium, derive_global_descriptors

SHOTS = (39915, 41524, 41672)
SIGNALS = {
    "plasma current": "magnetics.ip.0.data",
    "fill pressure": "barometry.gauge.0.pressure.data",
    "H-alpha": "spectrometer_uv.channel.0.processed_line.0.intensity.data",
}

shots = {shot: vaft.omas.load(vaft.data.sample(shot, representation="omas")) for shot in SHOTS}

rows = []
for shot, ods in shots.items():
    record = {"shot": shot, "IDS": len(ods.keys())}
    for name, path in SIGNALS.items():
        # .get answers without materializing the path it is asking about.
        value = ods.get(path, None)
        record[name] = "-" if value is None else f"{np.size(value)} samples"
    record["equilibrium"] = (
        f"{len(ods['equilibrium.time_slice'])} slices" if "equilibrium" in ods else "-"
    )
    rows.append(record)

inventory = pd.DataFrame(rows).set_index("shot")
print(inventory.to_string())

<repo>/vaft/omas/__init__.py:111: RuntimeWarning: Could not infer an IMAS DD version for omas_json; using 3.41.0


14:37:58 INFO     Parsing data dictionary version 3.41.0 @dd_zip.py:89


       IDS plasma current fill pressure       H-alpha equilibrium
shot                                                             
39915    9   2500 samples  2500 samples  2500 samples    9 slices
41524   11   2500 samples  2500 samples  2500 samples    6 slices
41672   11   2500 samples  2500 samples  2500 samples   19 slices


## Feature extraction plan

Every feature is derived by a VAFT API rather than by this page, so a pipeline
rule computing the same table gets the same numbers. Timing comes from
`plasma_timing`, which reports which detector decided the window and whether the
current and the light agree -- not a bare number. The equilibrium features come
from the descriptor layer, which leaves what it cannot derive unavailable.

In [ ]:
def features(shot, ods):
    """One row of the summary table, with its provenance."""
    timing = plasma_timing(ods)
    summary = timing.summary()
    row = {
        "shot": shot,
        "class": vaft.omas.classify_shot(ods),
        "onset_s": summary["onset"],
        "offset_s": summary["offset"],
        "duration_ms": None if not timing.found else (timing.offset - timing.onset) * 1e3,
        "timing_source": summary["source"],
        "timing_agreement": summary["agreement"],
    }

    if timing.ip is not None and timing.ip.found:
        # The accepted run is the pulse the detector kept; its evidence carries
        # both the peak and when it happened.
        accepted = timing.ip.onset.accepted
        row["ip_peak_kA"] = accepted.peak / 1e3
        row["ip_peak_time_s"] = accepted.peak_time
    else:
        row["ip_peak_kA"] = row["ip_peak_time_s"] = None

    row["b0_T"] = float(vaft.omas.find_bt(ods))

    pressure = np.asarray(ods["barometry.gauge.0.pressure.data"], dtype=float)
    row["fill_pressure_max"] = float(np.nanmax(pressure))
    halpha = np.asarray(
        ods["spectrometer_uv.channel.0.processed_line.0.intensity.data"], dtype=float
    )
    row["halpha_peak"] = float(np.nanmax(halpha))

    slices = ods["equilibrium.time_slice"]
    currents = [
        abs(float(slices[i]["global_quantities.ip"]))
        if "global_quantities" in slices[i] and "ip" in slices[i]["global_quantities"]
        else np.nan
        for i in range(len(slices))
    ]
    row["equilibrium_slices"] = len(slices)
    index = int(np.nanargmax(currents))
    descriptors = derive_global_descriptors(as_equilibrium(ods, time_index=index))
    for name, key in (("R0_m", "major_radius"), ("a_m", "minor_radius"),
                      ("kappa", "elongation"), ("q95", "q95"),
                      ("volume_m3", "volume"), ("beta_t", "beta_t"),
                      ("li", "li_virial")):
        item = descriptors[key]
        row[name] = float(item.value) if item.available else None
    return row


table = pd.DataFrame([features(shot, ods) for shot, ods in shots.items()]).set_index("shot")
print(table.to_string(float_format=lambda value: f"{value:.4g}"))

        class  onset_s  offset_s  duration_ms    timing_source timing_agreement  ip_peak_kA  ip_peak_time_s   b0_T  fill_pressure_max  halpha_peak  equilibrium_slices   R0_m    a_m  kappa   q95  volume_m3   beta_t     li
shot                                                                                                                                                                                                                        
39915  Plasma   0.3063    0.3308        24.48  h_alpha_primary       consistent       83.59          0.3141 0.1499            0.00433       0.9782                   9 0.3989 0.2949  1.536 8.671     0.9559 0.001274 0.6876
41524  Plasma   0.3146    0.3364        21.88  h_alpha_primary       consistent         222          0.3237 0.1734            0.00359        2.467                   6 0.3892 0.2852  1.619 4.247     0.9219 0.005092 0.6811
41672  Plasma   0.3122    0.3517        39.52  h_alpha_primary       consistent       127.2           0.328 0.1732  

`timing_source` says the H-alpha light decided every window here and
`timing_agreement` that the current agrees with it. A shot where they disagree
is exactly the shot a reviewer should open, which is what the review state below
is for.

## Classification criteria

`vaft.omas.classify_shot` decides from the shared plasma timing and the gas response: a plasma-current pulse found by `vaft.omas.plasma_timing` is `Plasma`; without one, a barometry pressure response or a window the light saw is `BD failure` (the shot was attempted); otherwise `Vacuum`. `vaft.omas.shot_class.shot_class` returns the record behind the string, with the check that decided. Only the pressure check has a threshold (`pressure_threshold`, a variance ratio); the light and the current are judged by the timing's detectors.

In [ ]:
from vaft.process.signal_processing import is_signal_active

print(f"{'shot':>7s} {'gas active':>11s} {'light active':>13s} {'max Ip [kA]':>12s}   class")
for shot, ods in shots.items():
    pressure = np.asarray(ods["barometry.gauge.0.pressure.data"], dtype=float)
    halpha = np.asarray(
        ods["spectrometer_uv.channel.0.processed_line.0.intensity.data"], dtype=float
    )
    current = np.asarray(ods["magnetics.ip.0.data"], dtype=float)
    print(f"{shot:>7d} {str(is_signal_active(pressure)):>11s} {str(is_signal_active(halpha)):>13s}"
          f" {np.nanmax(current) / 1e3:12.1f}   {vaft.omas.classify_shot(ods)}")

print()
print("threshold sweep (pressure_threshold):")
print(f"{'threshold':>10s} " + " ".join(f"{shot:>12d}" for shot in SHOTS))
for threshold in (1e-4, 1e-3, 1e-2, 1e-1, 0.5, 1.0):
    labels = [
        vaft.omas.classify_shot(ods, pressure_threshold=threshold)
        for ods in shots.values()
    ]
    print(f"{threshold:10.4g} " + " ".join(f"{label:>12s}" for label in labels))

   shot  gas active  light active  max Ip [kA]   class
  39915        True          True         84.1   Plasma
  41524        True          True        223.1   Plasma
  41672        True          True        127.0   Plasma

threshold sweep (pressure_threshold):
 threshold        39915        41524        41672
    0.0001       Plasma       Plasma       Plasma
     0.001       Plasma       Plasma       Plasma
      0.01       Plasma       Plasma       Plasma
       0.1       Plasma       Plasma       Plasma
       0.5       Plasma       Plasma       Plasma
         1       Plasma       Plasma       Plasma


All three hold their label from 1e-4 up to the 0.01 default, and then they part:
39915 reads as `Vacuum` at 0.1 while 41524 survives all the way to 1.0. That
spread is the shots themselves -- how strongly each one's gas and light traces
move relative to their own level -- not an artefact, and it is the reason a
campaign table has to record the threshold it used beside the label. The schema
below does.

## Manual review and label management

An automatic label is a proposal. The record keeps the proposal, the reviewer's
decision when there is one, and why -- so a corrected label never loses the
evidence it was corrected from. Nothing here is a new file format: these are
columns in the same table.

In [ ]:
REVIEW = {
    # shot: (reviewed_label, reviewer, note)
    41672: ("Plasma", "example-reviewer", "long pulse, checked against the camera"),
}

review = table[["class"]].rename(columns={"class": "auto_label"}).copy()
review["reviewed_label"] = [REVIEW.get(shot, (None,))[0] for shot in review.index]
review["review_state"] = np.where(review["reviewed_label"].isna(), "unreviewed", "reviewed")
review["reviewer"] = [REVIEW.get(shot, (None, None))[1] if shot in REVIEW else None
                      for shot in review.index]
review["note"] = [REVIEW.get(shot, (None, None, None))[2] if shot in REVIEW else None
                  for shot in review.index]
review["label"] = review["reviewed_label"].fillna(review["auto_label"])
review["disagrees"] = (
    review["reviewed_label"].notna() & (review["reviewed_label"] != review["auto_label"])
)
print(review.to_string())

      auto_label reviewed_label review_state          reviewer                                    note   label  disagrees
shot                                                                                                                     
39915     Plasma            NaN   unreviewed               NaN                                     NaN  Plasma      False
41524     Plasma            NaN   unreviewed               NaN                                     NaN  Plasma      False
41672     Plasma         Plasma     reviewed  example-reviewer  long pulse, checked against the camera  Plasma      False


## Summary spreadsheet schema

One row per shot, one column per feature, with the units and the provenance the
consumer needs to use a number without opening the shot. The file is written to
a temporary directory here; a pipeline rule writes it to its declared output.

In [ ]:
import tempfile
from pathlib import Path

SCHEMA = {
    "class": ("-", "vaft.omas.classify_shot"),
    "onset_s": ("s", "vaft.omas.plasma_timing"),
    "offset_s": ("s", "vaft.omas.plasma_timing"),
    "duration_ms": ("ms", "offset - onset"),
    "timing_source": ("-", "vaft.omas.plasma_timing"),
    "timing_agreement": ("-", "vaft.omas.plasma_timing"),
    "ip_peak_kA": ("kA", "plasma_timing ip window evidence"),
    "b0_T": ("T", "vaft.omas.find_bt"),
    "fill_pressure_max": ("a.u.", "barometry.gauge.0.pressure"),
    "halpha_peak": ("a.u.", "spectrometer_uv processed line"),
    "equilibrium_slices": ("-", "equilibrium.time_slice"),
    "R0_m": ("m", "derive_global_descriptors"),
    "a_m": ("m", "derive_global_descriptors"),
    "kappa": ("-", "derive_global_descriptors"),
    "q95": ("-", "derive_global_descriptors"),
    "volume_m3": ("m^3", "derive_global_descriptors"),
    "beta_t": ("-", "derive_global_descriptors"),
    "li": ("-", "derive_global_descriptors"),
}

summary = table.join(review[["label", "review_state", "reviewer", "note"]])
summary.insert(0, "classifier_threshold", 0.01)

print(f"{'column':22s} {'unit':8s} derived by")
for column, (unit, source) in SCHEMA.items():
    print(f"{column:22s} {unit:8s} {source}")

with tempfile.TemporaryDirectory() as scratch:
    target = Path(scratch) / "shot_summary.csv"
    summary.to_csv(target)
    print()
    print(f"{target.name}: {target.stat().st_size} bytes, {len(summary)} rows x {summary.shape[1]} columns")
    print(target.read_text().splitlines()[0][:200])

column                 unit     derived by
class                  -        vaft.omas.classify_shot
onset_s                s        vaft.omas.plasma_timing
offset_s               s        vaft.omas.plasma_timing
duration_ms            ms       offset - onset
timing_source          -        vaft.omas.plasma_timing
timing_agreement       -        vaft.omas.plasma_timing
ip_peak_kA             kA       plasma_timing ip window evidence
b0_T                   T        vaft.omas.find_bt
fill_pressure_max      a.u.     barometry.gauge.0.pressure
halpha_peak            a.u.     spectrometer_uv processed line
equilibrium_slices     -        equilibrium.time_slice
R0_m                   m        derive_global_descriptors
a_m                    m        derive_global_descriptors
kappa                  -        derive_global_descriptors
q95                    -        derive_global_descriptors
volume_m3              m^3      derive_global_descriptors
beta_t                 -        derive_global_de

## Campaign-level analysis

Three shots is not a campaign, and this page will not pretend otherwise: the
plot below is the shape of the analysis, not a result about VEST. The same
`features()` call over a database shot list produces the table a real campaign
view reads, which is what the gated cell after it does.

In [ ]:
fig, (ax_ip, ax_shape) = plt.subplots(1, 2, figsize=(11, 4.5))
ax_ip.scatter(table["duration_ms"], table["ip_peak_kA"], s=60)
for shot, row in table.iterrows():
    ax_ip.annotate(str(shot), (row["duration_ms"], row["ip_peak_kA"]),
                   textcoords="offset points", xytext=(6, 4), fontsize=8)
ax_ip.set(xlabel="pulse duration [ms]", ylabel="peak Ip [kA]", title="Discharge scale")

ax_shape.scatter(table["q95"], table["kappa"], s=60)
for shot, row in table.iterrows():
    ax_shape.annotate(str(shot), (row["q95"], row["kappa"]),
                      textcoords="offset points", xytext=(6, 4), fontsize=8)
ax_shape.set(xlabel="q95", ylabel="elongation", title="Equilibrium shape at peak current")

for ax in (ax_ip, ax_shape):
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

<tmp> UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [ ]:
import os

# The campaign path needs the VEST database. It is opt-in rather than attempted,
# so a reader without access gets a message instead of a timeout.
campaign = os.environ.get("VAFT_CAMPAIGN_SHOTS")
if not campaign:
    print("Set VAFT_CAMPAIGN_SHOTS to a comma-separated shot list to extend the table")
    print("over the database, for example:")
    print()
    print("    VAFT_CAMPAIGN_SHOTS=41500,41501,41502 jupyter nbconvert --execute ...")
else:
    requested = [int(item) for item in campaign.split(",") if item.strip()]
    collected = []
    for shot in requested:
        try:
            collected.append(features(shot, vaft.database.open(shot)))
        except Exception as error:  # one unreadable shot must not end the campaign
            print(f"{shot}: skipped -- {type(error).__name__}: {error}")
    if collected:
        print(pd.DataFrame(collected).set_index("shot").to_string(
            float_format=lambda value: f"{value:.4g}"))

Set VAFT_CAMPAIGN_SHOTS to a comma-separated shot list to extend the table
over the database, for example:

    VAFT_CAMPAIGN_SHOTS=41500,41501,41502 jupyter nbconvert --execute ...


## Snakemake aggregation rules

The shape this notebook implies for the pipeline: one rule per shot producing a
feature record, one aggregation rule concatenating them into the summary table.
The per-shot rule is `features()` above; the aggregation is a concatenation and
nothing more, which is what makes it safe to re-run for a single changed shot.

```text
rule shot_features:
    input:  "processed/{shot}/ods.json"
    output: "features/{shot}.json"

rule shot_summary:
    input:  expand("features/{shot}.json", shot=CAMPAIGN)
    output: "summary/shot_characteristics.csv"
```

The review columns are the exception: they are edited by people, so an
aggregation rule joins them from a tracked file rather than regenerating them.

## What this page establishes

- A per-shot feature record built entirely from VAFT APIs, so a pipeline rule
  and a notebook produce the same numbers.
- A class label with its inputs and its threshold sensitivity shown, rather than
  asserted.
- A review state that keeps the automatic proposal beside the human decision.
- A summary schema with units and provenance per column, written as the table an
  aggregation rule would produce.

Three packaged shots are enough to define the record and check it end to end;
they are not a campaign, and no trend here should be read as one.